# Rust 108: Assertions and Type Checks

Chapter 08 of the Rust track, and the one about *proving things while you run — and while you compile*. Chapters 01–07 built values, functions, ownership and the I/O boundary; this chapter is the checking toolkit that keeps them honest: `assert!` and its family, `debug_assert!`, `matches!`, runtime type inspection through `std::any`, and `size_of` for measuring what the compiler laid out. It is also the shortest arc in the series — because half of "checking" in Rust is not runtime work at all, and the chapter's job is to show you which half is which.

Everything here is a **runnable example**. Read the markdown, run the cell below it, and change nothing — the practice twin of this notebook is `08_assertions-and-type-checks_p.ipynb`.

## Who this is written for

You are assumed to be a competent programmer, arriving from Python and C++. You have written `assert` statements, unit tests and `isinstance` checks before — none of that is new. What is new: an `assert!` failure is a **panic with a structured payload**, not an exception to catch by type; the same assertion can run at **compile time** with one extra keyword; and the type system itself is the strongest assertion in the language — several of this chapter's tools exist to tell you what the compiler already proved for free.

## How to read this notebook

- Markdown cells explain; code cells demonstrate. **Run each code cell as you reach it.** Cells are independent — there is no sandbox and no shared state beyond the kernel namespace, so a restart-and-run-from-the-top always works.
- Code that intentionally *fails* is shown as a comment, together with the diagnostic or panic message you would get. §8 is the catalogue.
- One cell (§1.3) deliberately catches a panic; every other cell exits green.
- The prose is terse and the facts are dense. This is a reference you will come back to, not a novel you read once.

## The one idea to internalise

**Rust expresses contracts at two scales, and the compiler is the cheaper scale.** A runtime assertion costs cycles on every execution and pays off only when it fires; a compile-time fact — a type, a `const` assertion, a pattern the match checker exhausts — is enforced *always*, for free, before the program ever runs. The chapter's habit: when you reach for `assert!`, ask whether a `const` block, a richer type, or `matches!` at the boundary could move that check to the compiler. Runtime assertions are for facts about *data*; compile-time checks are for facts about *the program*.


### Contents

- [0. Notebook Conventions: How These Cells Run](#0-notebook-conventions-how-these-cells-run)
- [1. The Assertion Family: Panic as a Feature](#1-the-assertion-family-panic-as-a-feature)
- [2. Assertions at Compile Time](#2-assertions-at-compile-time)
- [3. `debug_assert!`: Checks You Can Compile Away](#3-debug_assert-checks-you-can-compile-away)
- [4. `matches!`: A Pattern Test, Not a Match](#4-matches-a-pattern-test-not-a-match)
- [5. Runtime Type Inspection: `std::any`](#5-runtime-type-inspection-stdany)
- [6. Size of Things: `size_of` and Friends](#6-size-of-things-size_of-and-friends)
- [7. What Is Checkable When: Runtime vs Compile Time](#7-what-is-checkable-when-runtime-vs-compile-time)
- [8. Reading the Compiler: Panics and E-Codes](#8-reading-the-compiler-panics-and-e-codes)
- [9. Performance Notes for Low-Latency Code](#9-performance-notes-for-low-latency-code)
- [10. Mental Model: Contracts at Two Scales](#10-mental-model-contracts-at-two-scales)
- [11. Habits and Review](#11-habits-and-review)
- [12. Cheat Sheet](#12-cheat-sheet)
- [13. Review Questions](#13-review-questions)

If you are returning to this chapter later, §12 is the one-page summary, §8 is the diagnostics catalogue, and §9 is where the latency notes live.


## 0. Notebook Conventions: How These Cells Run

This notebook runs on the **evcxr** kernel, which wraps each cell into a generated program and compiles it incrementally. Three consequences matter for every cell below:

- **There is no `fn main`.** The kernel supplies one; you are typing its body.
- **State persists across cells.** A `let` from cell 4 is still live in cell 20 — including `use` statements.
- **Items are order-independent, statements are not.** Two `fn`s may call each other regardless of order; two `let`s may not use each other's names before they run.

There is **one namespace per notebook**. Two items with the same name collide exactly as they would in one source file, even in different sections. If you re-run a cell that defines an item, it is redefined; on a redefinition error, restart the kernel and run from the top.

A failing cell does not poison the kernel — the deliberate compile errors in this chapter are comments, so every cell still runs. §1.3 catches its own panic with `std::panic::catch_unwind`, so even that cell exits green.

### A chapter without a sandbox

Unlike chapter 07, this notebook touches **no files**: assertions, type checks and size queries are in-process by nature. That is itself a lesson — everything here runs in the address space the compiler already reason about, which is exactly why most of it can be moved to compile time.


In [ ]:
// 0.1 Smoke test: an assertion passes, a value comes back, the kernel is alive.
let spread = 4131.0 - 4125.5;
assert!(spread > 0.0, "spread must be positive, got {spread}");
println!("kernel alive, spread = {spread}");


## 1. The Assertion Family: Panic as a Feature

The three runtime macros form one family: `assert!(cond)` checks a boolean; `assert_eq!(a, b)` and `assert_ne!(a, b)` check equality and inequality *and print both values* when the check fails. All of them **panic** on failure — and in Rust, a panic is a structured feature: an unwind carrying a *payload* you can catch and inspect (§1.3), not an untyped exception.

| Macro | Fires when | Failure payload highlights |
|---|---|---|
| `assert!(cond, msg..)` | `cond` is false | your message, formatted with the args |
| `assert_eq!(a, b)` | `a != b` | ``assertion `left == right` failed`` + both values |
| `assert_ne!(a, b)` | `a == b` | ``assertion `left != right` failed`` + both values |

Two obligations come with the family: the condition must be a real check (not a side effect), and **`assert_eq!`/`assert_ne!` require both operands to implement `Debug`** — that is how they print the failure report (§8 shows the compiler error when a type does not).

### Assertions on real data


In [ ]:
// 1.1 assert! guards an invariant over a small order book.
let bids = [4124.5, 4125.0, 4125.5];
let asks = [4126.0, 4126.5];
let best_bid = bids[bids.len() - 1];
let best_ask = asks[0];
assert!(best_bid < best_ask, "crossed book: bid {best_bid} >= ask {best_ask}");
assert!(bids.is_sorted(), "bids must ascend");
println!("book sane: bid {best_bid} < ask {best_ask}");


### The report is the payload

`assert_eq!` builds its failure report from `Debug` — one line naming the macro's contract, then `left:` and `right:` values. The report is worth reading closely because the *values* are `{:?}`-formatted: quoted strings, full floats, containers with their contents.

### A failing report, caught and shown


In [ ]:
// 1.2 The assert_eq! failure report, captured instead of crashing the cell.
use std::panic;
let reported = panic::catch_unwind(|| {
    assert_eq!(4.0f64, 5.0f64); // would panic: left: 4.0, right: 5.0
});
let payload = reported.unwrap_err();
let text = payload.downcast_ref::<String>().cloned().unwrap_or_default();
println!("payload = {text:?}");
assert!(text.contains("assertion `left == right` failed"));
assert!(text.contains("left: 4.0") && text.contains("right: 5.0"));


The payload from `assert_eq!` downcasts to an owned `String` — the macro formats the report eagerly. The same is **not** true of `assert!` with a format string, which is the chapter's first counter-intuitive fact.

### The payload taxonomy: who builds what

Probed on this exact toolchain (rustc 1.98, edition 2024), the payload types split three ways:

| Panic source | Payload downcasts to | Why |
|---|---|---|
| `assert!(false, "bare")` | `&str` | a literal message needs no formatting work |
| `assert!(false, "{}", x)` | `&str` | the **formatting is deferred** until the panic handler prints it — the payload is the unevaluated `Arguments` |
| `panic!("literal")` | `&str` | same lazy mechanism |
| `assert_eq!(a, b)` | `String` | the two-value report is built eagerly so both `Debug` values render |

The practical lesson: **generic payload extraction needs both downcasts** — try `String`, then `&str` — and the order does not matter because only one of them can hit.


In [ ]:
// 1.3 One extractor for all payloads: String first, then &str.
// (std::panic is already in scope from 1.2)
/// Pull a human-readable message out of any panic payload.
fn panic_text(payload: Box<dyn std::any::Any + Send>) -> String {
    payload.downcast_ref::<String>().cloned()
        .or_else(|| payload.downcast_ref::<&str>().map(|s| s.to_string()))
        .unwrap_or_else(|| "<opaque payload>".to_string())
}
let from_eq = panic::catch_unwind(|| assert_eq!(2 + 2, 5)).unwrap_err();
let from_assert = panic::catch_unwind(|| panic!("plain literal")).unwrap_err();
let eq_text = panic_text(from_eq);
let plain_text = panic_text(from_assert);
println!("assert_eq! -> {eq_text:?}");
println!("panic!     -> {plain_text:?}");
assert!(eq_text.contains("left == right"));
assert_eq!(plain_text, "plain literal");


## 2. Assertions at Compile Time

`assert!` works in **const contexts** — `const` items, `static` initialisers, array-length expressions. A const-context assertion is not a runtime check at all: it is evaluated during compilation, and a failure is a **compile error** (E0080, "evaluation panicked"), not a panic. This is the cheapest assertion in the language: it costs nothing at runtime, and it can only fire once, on your build machine.

The classic uses: pinning layout assumptions and validating arithmetic the compiler will otherwise do blindly. The idiom is `const _: () = assert!(..);` — the `_` name means "evaluate and discard".

### A layout contract as a compile error


In [ ]:
// 2.1 A const assert pins a layout fact: breaking it is a compile error.
/// A fixed-layout record; the const asserts below are its contract.
struct Order {
    id: u64,     // 8
    qty: u32,    // 4
    flags: u32,  // 4  (the compiler would pad here anyway; this field earns the slot)
    price: u64,  // 8
    symbol: u64, // 8
}                // = 32 bytes with no hidden padding
const _: () = assert!(size_of::<Order>() == 32, "Order layout drifted");
const _: () = assert!(
    size_of::<Option<&Order>>() == size_of::<&Order>(),
    "references carry a niche, so Option<&T> is free"
);
let o = Order { id: 7, qty: 300, flags: 0b101, price: 412_550, symbol: 10_001 };
let checksum = o.id ^ u64::from(o.qty) ^ u64::from(o.flags) ^ o.price ^ o.symbol;
println!("layout held: Order is {} bytes (checksum {checksum})", size_of::<Order>());


If a future edit pushed `Order` past 32 bytes, the build would stop with the exact diagnostic this toolchain produces for a deliberately false const assert:

```text
error[E0080]: evaluation panicked: Order layout drifted
```

No runtime cost, no test to remember to run — the contract is enforced every time anything in the file compiles. Note the second assert: `Option<&Order>` measuring the same as `&Order` is the *niche* fact §6 will tabulate — here it is pinned so a toolchain change cannot silently inflate every `Option<&T>` in the codebase.

### An arithmetic invariant, checked before main


In [ ]:
// 2.2 A const table plus the invariants and derivations that hang off it.
const TICKS: [u64; 4] = [4124, 4125, 4126, 4127];
const _: () = assert!(TICKS.len() > 0, "tick table must not be empty");
const _: () = assert!(TICKS[0] < TICKS[TICKS.len() - 1], "tick table must ascend");
const MIDPOINT: u64 = (TICKS[0] + TICKS[TICKS.len() - 1]) / 2;
/// Distance from the table's midpoint — the table's contract guarantees it exists.
fn midpoint_distance(price: u64) -> u64 {
    price.abs_diff(MIDPOINT)
}
println!("midpoint {MIDPOINT}, distance of 4131 = {}", midpoint_distance(4131));
assert_eq!(midpoint_distance(4131), 6);


## 3. `debug_assert!`: Checks You Can Compile Away

`debug_assert!`, `debug_assert_eq!` and `debug_assert_ne!` are the same three macros with one difference: they compile to **nothing** unless the build has debug assertions on. This kernel builds with `cfg!(debug_assertions) == true`, so they run — but the same source compiled with `-O` would carry no trace of them. The division of labour:

| Check | Runs in | Use for |
|---|---|---|
| `assert!` family | every build | invariants whose violation is a bug *anytime* |
| `debug_assert!` family | debug builds only | internal invariants too costly per-tick; release correctness comes from types |

The HFT framing: an order-book loop can afford `debug_assert!`s that pin its structural assumptions during development, then ship release binaries where the checks are gone and the *types* (non-negative quantities, sorted ladders as distinct types) carry the guarantee.

### Live in this kernel, gone in release


In [ ]:
// 3.1 debug_assert! is live in this kernel: the build has debug assertions on.
let ladder = [4125, 4126, 4127];
debug_assert!(ladder.first() < ladder.last(), "ladder must ascend");
assert!(cfg!(debug_assertions), "this kernel builds with debug assertions");
println!("cfg!(debug_assertions) = {}", cfg!(debug_assertions));
// Compiled with -O (release), this same check would vanish from the binary:
// debug_assert! compiles to nothing unless debug_assertions is on.


### One that fires — caught, so the cell stays green


In [ ]:
// 3.2 A debug_assert! that fires — caught, so the cell stays green.
let caught = panic::catch_unwind(|| {
    let book = vec!["ask-only"];
    debug_assert!(book.len() > 1, "two-sided book required");
});
let text = panic_text(caught.unwrap_err());
println!("debug_assert fired in debug mode: {text:?}");
assert!(text.contains("two-sided book required"));


`panic_text` is §1.3's extractor — items are order-independent, so later cells reuse it freely. Under the kernel the assert fired because debug assertions are on; in a `-O` build the same cell would print nothing at all, which is exactly the deal `debug_assert!` offers.

## 4. `matches!`: A Pattern Test, Not a Match

`matches!(expr, pattern)` evaluates to `true` when `expr` matches the pattern — a `match` expression collapsed to a boolean, with the same pattern power: `..` rest patterns, bindings, guards (`if`), or-patterns. It is the right tool *inside* assertions, because it asserts the shape you mean instead of re-deriving it with accessors.

| Written as | Instead of |
|---|---|
| `assert!(matches!(fill, Fill::Partial { .. }))` | `assert!(matches!(fill.kind(), ...))` accessor chains |
| `matches!(res, Ok(0) \| Err(_))` | two `is_ok_and`/`map_err` calls |
| `matches!(opt, Some(v) if v > 0)` | `opt.map_or(false, \|v\| v > 0)` |

### Pattern power in a boolean


In [ ]:
// 4.1 matches! with rest patterns, bindings, guards and or-patterns.
#[derive(Debug)]
enum Fill {
    Full { qty: u32 },
    Partial { qty: u32, remaining: u32 },
}
let fills = [Fill::Full { qty: 300 }, Fill::Partial { qty: 300, remaining: 120 }];
for f in &fills {
    match f {
        Fill::Full { qty } => println!("full fill for {qty}"),
        Fill::Partial { qty, remaining } => println!("partial {qty}, {remaining} left"),
    }
}
assert!(matches!(fills[0], Fill::Full { .. }));
assert!(!matches!(fills[0], Fill::Partial { .. }));
assert!(matches!(fills[1], Fill::Partial { remaining, .. } if remaining > 0));
let side: Result<u32, String> = Err("rejected".to_string());
assert!(matches!(side, Ok(0) | Err(_)));
println!("all four match shapes verified");


The guard arm deserves the second look: `matches!(fills[1], Fill::Partial { remaining, .. } if remaining > 0)` both binds and tests — the `..` keeps the pattern honest about what it ignores. When the check grows beyond a shape test (extracting the value, acting on it), that is the signal to leave `matches!` for a real `match`; the macro is a boolean, not a control-flow tool.


## 5. Runtime Type Inspection: `std::any`

Most of the time, types are compile-time facts — and the *next* section is about measuring them. But a few designs genuinely need to ask "what is this value?" while running: `dyn Trait` objects, panic payloads, type-keyed caches. `std::any` provides three tools at three levels of power:

| Tool | Gives you | Cost | Caveat |
|---|---|---|---|
| `type_name::<T>()` | a human-readable string | compile-time; free at runtime | **descriptive only** — never compare it |
| `TypeId::of::<T>()` | an equality-comparable ID | zero-size, `Copy`, const-constructible | `'static` types only |
| `downcast_ref::<T>()` | `Option<&T>` from a `dyn Any` | one `TypeId` compare | the `Any` value must actually be a `T` |

`type_name` is the debugging tool: `type_name::<i32>()` is `"i32"`, and generics print their full path (`alloc::vec::Vec<(alloc::string::String, f64)>`). Because it is just a string, it is for *logs and panic messages* — two types with the same name in different crates compare equal as strings while being different types, which is why `TypeId` exists.

### Naming types for humans


In [ ]:
// 5.1 type_name: readable names, full generic paths, honest about being text.
use std::any::type_name;
fn describe<T>(values: &[T]) -> String {
    format!("{} elements of {}", values.len(), type_name::<T>())
}
println!("{}", describe(&[4125.5f64, 4126.0, 4126.5]));
println!("{}", describe(&["AAPL", "MSFT"]));
assert_eq!(type_name::<i32>(), "i32");
let _ = type_name::<Vec<(String, f64)>>(); // full path in a panic message is its day job


`type_name` is **not** in the prelude — `size_of` is, `type_name` needs its `use` (the probe proved both directions). It also cannot be called with `T` whose size is not known without a turbofish in some positions; `describe` above dodges that by taking `&[T]`.

### `TypeId`: identity, not description

`TypeId::of::<T>()` is a zero-sized, `Copy`, hashable token that compares equal only for the same type. It is what `HashMap<TypeKey, V>` caches and what `dyn Any` downcasts are built on. Two rules: the type must be `'static` (no borrows), and *you* never compare `type_name` strings where `TypeId` is available.

### Identity checks across a generic boundary


In [ ]:
// 5.2 TypeId: equal for the same type, different across wrappers.
use std::any::TypeId;
/// Two tags agree only when they name the same type — checked with TypeId.
fn same_type<A: 'static, B: 'static>() -> bool {
    TypeId::of::<A>() == TypeId::of::<B>()
}
assert!(same_type::<u32, u32>());
assert!(!same_type::<u32, i32>());
assert!(!same_type::<Vec<u8>, Vec<i8>>()); // same shape, different element type
println!("TypeId identity verified");


### `dyn Any` and downcasting

`Box<dyn Any>` is a value whose concrete type the compiler has *erased*; `downcast_ref::<T>()` asks "are you a `T`?" and answers with `Option<&T>`. This is the machinery under `panic::catch_unwind` (§1.3) — the payload is `Box<dyn Any + Send>`, and the two downcasts there are exactly this section at work. The honest framing: **downcasting is the tool for values that cross an erasure boundary** (panic payloads, plugin registries, `Box<dyn Error>`'s own `downcast_ref`). Inside normal code, generics and enums need none of this — the compiler knew the type all along.

### The erasure boundary, exercised


In [ ]:
// 5.3 Downcast across an erasure boundary: the panic-payload pattern.
use std::any::Any;
/// Accept any payload; report what it really is.
fn identify(payload: &dyn Any) -> String {
    if let Some(n) = payload.downcast_ref::<i64>() {
        format!("i64: {n}")
    } else if let Some(s) = payload.downcast_ref::<&str>() {
        format!("&str: {s}")
    } else {
        "something else".to_string()
    }
}
let typed: Box<dyn Any> = Box::new(9_500_000_000i64);
let texty: Box<dyn Any> = Box::new("replay gap");
println!("{}", identify(&*typed));
println!("{}", identify(&*texty));
assert!(typed.downcast_ref::<f64>().is_none());
assert!(texty.downcast_ref::<&str>().is_some());


## 6. Size of Things: `size_of` and Friends

`size_of::<T>()` returns the size of `T` **in bytes** — and it is a `const` function that is also **in the prelude**: no import needed (a deliberate contrast with `type_name`, which needs one). Its sibling `size_of_val(&v)` measures a *value* — for a slice `&[u8]` that means the pointed-to bytes, not the fat pointer.

### The niche: where `Option` is free — and where it is not

The table every Rust programmer should be able to recite (probed on this toolchain):

| Type | `size_of` | Why |
|---|---|---|
| `()` | 0 | zero-sized type |
| `i32` | 4 | — |
| `Option<i32>` | **8** | `i32` has all bit patterns taken — the discriminant needs its own byte-word |
| `&i32` | 8 | a pointer |
| `Option<&i32>` | **8** | `0` is not a valid reference → **niche**: `None` hides in the pointer |
| `Box<u8>` | 8 | — |
| `Option<Box<u8>>` | **8** | same niche trick |
| `String` / `Vec<i32>` | 24 | ptr + cap + len (three machine words) |
| `&str` / `&[u8]` | 16 | ptr + len (fat pointer, no capacity) |

The lesson cuts both ways: wrapping a plain integer in `Option` **doubles** it, while wrapping a reference or `Box` costs **nothing**. In hot structures, that decides whether a field is `Option<i32>` or a sentinel value.

### Measuring the table live


In [ ]:
// 6.1 The size table, verified live: niches make Option free exactly where promised.
assert_eq!(size_of::<()>(), 0);
assert_eq!(size_of::<i32>(), 4);
assert_eq!(size_of::<Option<i32>>(), 8);      // no niche: discriminant added
assert_eq!(size_of::<&i32>(), 8);
assert_eq!(size_of::<Option<&i32>>(), 8);     // niche: same size
assert_eq!(size_of::<Option<Box<u8>>>(), size_of::<Box<u8>>()); // niche again
assert_eq!(size_of::<String>(), 24);
assert_eq!(size_of::<Vec<i32>>(), 24);
assert_eq!(size_of::<&str>(), 16);
println!("String = {} bytes, &str = {} bytes", size_of::<String>(), size_of::<&str>());


### `size_of_val`: measuring what is pointed at

`size_of::<&[u8]>()` is 16 — the fat pointer. `size_of_val(slice)` is the *pointed-to bytes*: for `[0u8; 10]` viewed as a slice, 10. The distinction is "size of the handle" vs "size of the thing", and it matters most for slices and `dyn Trait` references, where the handle and the payload differ.

### Handle versus payload


In [ ]:
// 6.2 size_of the handle vs size_of_val the payload.
let bytes = [0u8; 10];
let view: &[u8] = &bytes;
assert_eq!(size_of::<&[u8]>(), 16);        // the fat pointer
assert_eq!(size_of_val(&view[..]), 10);    // the bytes it points at
let units = vec![(); 3];
assert_eq!(size_of::<()>(), 0);
assert_eq!(units.len(), 3);
println!("handle = {} bytes, payload = {} bytes",
    size_of::<&[u8]>(), size_of_val(view));


### Sizes are const facts too

`size_of` works in `const` contexts — §2.1's layout contract was built on it. One more placement rule worth pinning: a `const` item can compute with `size_of` at compile time, so "this buffer is at least a cache line" is expressible as a checked constant:

### A compile-time sizing gate


In [ ]:
// 6.3 Const-context sizing: the ring buffer slot must fit a cache line.
/// One ring-buffer slot; sized so two fit in 64 bytes with room for a flag.
struct Slot {
    price: u64,
    qty: u32,
    seq: u32,
}
const _: () = assert!(size_of::<Slot>() == 16, "Slot must stay 16 bytes");
const SLOTS_PER_LINE: usize = 64 / size_of::<Slot>();
const _: () = assert!(SLOTS_PER_LINE == 4, "packing changed");
let slot = Slot { price: 412_550, qty: 300, seq: 1 };
println!(
    "Slot = {} bytes -> {SLOTS_PER_LINE} per 64-byte line (slot: price {}, qty {}, seq {})",
    size_of::<Slot>(), slot.price, slot.qty, slot.seq
);


## 7. What Is Checkable When: Runtime vs Compile Time

The chapter's organizing question, answered as a table. The left column lists checks you might be tempted to write with `assert!`; the right column names where each *can* live — and the cheapest home is the one to choose.

| Check | Compile time possible? | The compile-time tool | Runtime fallback |
|---|---|---|---|
| type of a value is what a branch assumes | yes (usually) | the type system; `matches!` at the boundary | `dyn Any` downcast (§5.3) |
| two types are the same | yes | generics — one `T` *is* the check | `TypeId::of::<A>() == TypeId::of::<B>()` (§5.2) |
| struct layout is stable | yes | `const _: () = assert!(size_of ..)` (§2.1) | none needed |
| arithmetic invariant of constants | yes | const `assert!` (§2.2) | `assert!` in `main`'s setup |
| collection is sorted / book is crossed | no — it is *data* | — | `assert!`/`debug_assert!` (§1, §3) |
| parsed value is in range | partly | `enum` with fixed variants; `try_from` in a constructor | `assert!` at the edge |
| a file was written correctly | no | — | `assert_eq!` on read-back (ch 07) |

The pattern behind the table: **facts about the program go to the compiler; facts about the data stay at runtime.** "The ladder is sorted" depends on values that arrive from outside — no compile-time tool can know it, so the assert is honest runtime work. "The layout is 32 bytes" is a property of the program's own text — a runtime check for it would be an admission that a compile-time one was available and skipped.

### The same fact, both scales — side by side


In [ ]:
// 7.1 One invariant expressed twice: const gate + runtime guard.
/// Quotes must arrive in ascending time order; the fixed sample table is compile-time.
const SAMPLE_TS: [u64; 4] = [1, 4, 9, 16];
const ASCENDING: bool = {
    let mut ok = true;
    let mut i = 1;
    while i < SAMPLE_TS.len() {
        ok = ok && SAMPLE_TS[i - 1] < SAMPLE_TS[i];
        i += 1;
    }
    ok
};
const _: () = assert!(ASCENDING, "sample ts must ascend");
/// The runtime twin: guard the *data* the same way with debug_assert.
fn replay(samples: &[u64]) {
    debug_assert!(
        samples.windows(2).all(|w| w[0] < w[1]),
        "replay stream must ascend"
    );
}
replay(&SAMPLE_TS);
replay(&[7, 8, 9]);
println!("both scales checked; a descending slice would trip the debug_assert");
// replay(&[9, 8]); // <- debug build panics; release compiles it away (see §3)


Cell 7.1 is the chapter in one cell: the same ascending-order invariant guards a `const` table (compile time, free forever) and a runtime slice (`debug_assert!`, free in release). Writing the runtime check was not a failure to be cleverer — stream data genuinely arrives at run time.

The const arm also carries a sharp edge the probe caught while writing this notebook: `SAMPLE_TS.windows(2).all(..)` **does not compile in a const context** — `windows` and `Iterator::all` are not `const fn`s (E0015). Const eval accepts `while` loops and local `let`s, so the const twin is spelled as the `ASCENDING` loop above. The predicate can be identical in spirit; the *methods available* differ between the two scales, and the compiler will tell you with E0015 which side of the line a call sits on.

## 8. Reading the Compiler: Panics and E-Codes

This chapter's diagnostics are mostly about what assertions *demand* of your types. Every wording below was captured from this exact toolchain.

| Diagnostic | Trigger | Message (abridged) | Fix |
|---|---|---|---|
| **E0080** | `const _: () = assert!(1 + 1 == 3)` | ``evaluation panicked: math broke`` | the const-evaluated condition is false — the "panic" happened at compile time |
| **E0369 + E0277 ×2** | `assert_eq!(a, b)` on a bare struct | ``binary operation `==` cannot be applied`` *and* `` `Price` doesn't implement `Debug` `` (twice, left and right) | `#[derive(PartialEq, Debug)]` — `assert_eq!` needs **both** |
| **E0308** | `assert!(x)` with a non-bool `x` | ``expected `bool`, found integer`` | the condition really must be a boolean; `x != 0` or `matches!` |
| **E0599** | `TypeId::of_value(&v)` | ``no associated function ... found`` | the from-a-value constructor is not stable — use `TypeId::of::<T>()` |

Three detail worth keeping: the triple-error shape of a `Debug`-less `assert_eq!` (the equality bound *and* a `Debug` bound for each side) surprises everyone exactly once; and a const assert failure points at the *item*, with "erroneous constant encountered" where it was used.

### Runtime failure wording, catalogue section


In [ ]:
// 8.1 The failure wordings, verified by catching them.
let eq = panic_text(panic::catch_unwind(|| assert_eq!(1, 2)).unwrap_err());
let ne = panic_text(panic::catch_unwind(|| assert_ne!(2, 2)).unwrap_err());
let plain = panic_text(panic::catch_unwind(|| assert!(false, "invariant violated")).unwrap_err());
println!("eq    = {eq:?}");
println!("ne    = {ne:?}");
println!("plain = {plain:?}");
assert!(eq.contains("assertion `left == right` failed") && eq.contains("left: 1"));
assert!(ne.contains("assertion `left != right` failed") && ne.contains("right: 2"));
assert!(plain.contains("invariant violated"));


## 9. Performance Notes for Low-Latency Code

Assertions are *free until they fire* only in the branch-predicted sense — each check is still instructions in the hot path. The chapter's cost rules:

| Habit | Cost shape | Fix |
|---|---|---|
| `assert!` in a per-tick loop, release builds included | predictable branch + rare panic path, every iteration | `debug_assert!` when the check is a development guard |
| formatting args in `assert!(false, "{}", x)` | **no cost on success** — the format work is part of the panic path only | write rich messages freely |
| `assert_eq!` on `String`/`Vec` operands | success path is a length+byte compare; failure pays the report | fine for config paths; be deliberate per-tick |
| `TypeId`/`downcast_ref` in a hot dispatch loop | one compare; but a `match` on your own enum is cheaper *and* exhaustive | enums for closed sets; `Any` for open ones |
| `Option<i32>` fields chosen without measuring | doubled field size (§6), cache-line pressure | niche types or sentinels |

Two low-latency habits worth internalising: **put the invariant in the type when the check would run per-tick** (chapter 10's newtypes are the mechanism), and **keep the `debug_assert!`s anyway** — they document the assumption and cost nothing in release.

### The branching cost, measured


In [ ]:
// 9.1 A passing assert! is not free: it is a compare + branch in the loop.
use std::time::Instant;
const N: u32 = 20_000_000;
let t0 = Instant::now();
let mut acc: u64 = 0;
for i in 0..N {
    let v = u64::from(i & 0xFF);
    acc = acc.wrapping_add(v);
}
let bare = t0.elapsed();
let t0 = Instant::now();
let mut acc2: u64 = 0;
for i in 0..N {
    let v = u64::from(i & 0xFF);
    assert!(v <= 255, "impossible");
    acc2 = acc2.wrapping_add(v);
}
let guarded = t0.elapsed();
println!("bare loop {bare:?}, assert-guarded {guarded:?}");
assert_eq!(acc, acc2);


The numbers wander run to run (branch prediction is good) — the lesson is not the exact ratio but the shape: a passing `assert!` is real work in the loop body, while `debug_assert!` in release is *zero* work. That is the whole trade §3 described, now with a stopwatch attached.

## 10. Mental Model: Contracts at Two Scales

| Scale | Tool | When it fires | Cost when it holds |
|---|---|---|---|
| the program's own facts | types, `const` assert, exhaustive `match` | at compile time | zero |
| data arriving at run time | `assert!` family, `debug_assert!`, `matches!` | when the invariant breaks | a compare + branch |
| values behind an erasure boundary | `dyn Any` downcasts, `TypeId` | when you ask | one compare + hash setup where applicable |

Read the middle row's "when it fires" carefully: a runtime assertion that never fires in testing has bought you nothing yet — its value is *conditional on the bad input arriving*. The compile-time row has no such condition: once it compiles, the fact is settled. That asymmetry is why the chapter's habit list leads with "move it to the compiler if you can".

## 11. Habits and Review

- **Prefer the compiler:** for every `assert!`, ask whether a type, a const gate or `matches!` at the boundary could take the job (§7).
- **`assert_eq!` needs `PartialEq` + `Debug`** — derive both, and the failure report writes itself (§8).
- **`debug_assert!` for internal invariants in hot paths;** keep them as executable documentation; release correctness comes from types (§3, §9).
- **Write the failure message with the values in it** — formatting costs nothing on success (§9).
- **`matches!` for shape tests, real `match` for anything that extracts or acts** (§4).
- **`type_name` for humans, `TypeId` for identity** — never compare name strings (§5).
- **Recite the niche table:** `Option<reference>` is free, `Option<plain integer>` is not (§6).
- **Downcast both payload types** when catching panics: `String` then `&str` (§1.3).

## 12. Cheat Sheet

| Task | Code |
|---|---|
| boolean guard with context | `assert!(cond, "msg {x}")` |
| equality with report | `assert_eq!(a, b)` / `assert_ne!(a, b)` |
| dev-only guard | `debug_assert!(cond)` (gone in release) |
| which build is this | `cfg!(debug_assertions)` |
| shape test as bool | `matches!(v, Pat \| Pat if guard)` |
| name a type (display only) | `std::any::type_name::<T>()` (needs `use`) |
| type identity | `TypeId::of::<A>() == TypeId::of::<B>()` |
| ask a `dyn Any` | `v.downcast_ref::<T>() -> Option<&T>` |
| size of a type | `size_of::<T>()` (prelude, const-usable) |
| size of a value | `size_of_val(&v)` |
| compile-time gate | `const _: () = assert!(.., "msg");` |
| catch + read a panic | `panic::catch_unwind(f)` then downcast `String`/`&str` |

## 13. Review Questions

Answer from memory; each one is a §-numbered cell away.

1. Why does `assert_eq!` demand `Debug` on both operands, while `assert!` demands nothing? (§1, §8)
2. Your const assert on `size_of::<Order>()` just broke the build. What kind of error is this, where did it "panic", and what is the exit path? (§2)
3. In a release build, which of the chapter's checks still run, and what replaced the ones that do not? (§3, §7)
4. `matches!(res, Ok(0) | Err(_))` — why is this clearer than `res.is_ok_and(|v| v == 0) || res.is_err()`? What pattern feature is doing the work? (§4)
5. `type_name::<T>()` returns `"core::option::Option<u8>"` for two different crates' `Option`-like types. Why must you not compare those strings, and what do you use instead? (§5)
6. `Option<u32>` is 8 bytes but `Option<&u32>` is 8 bytes too — same number, opposite stories. Tell both. (§6)
7. Where does `size_of_val(&slice)` differ from `size_of::<&[T]>()`, and what exactly does each measure? (§6)
8. A hot loop guards an internal invariant with `assert!`. Give the two-step argument for changing it to `debug_assert!` and what must then carry the guarantee in release. (§3, §7, §9)

### What Comes Next

Chapter 08 hands you the checking toolkit and the habit of choosing the cheapest scale for every contract. The core track has one chapter left before the crate world begins.

| Chapter | Title | What it settles |
|---|---|---|
| **09** | **Logging** | `tracing` events, spans and levels — the structured, filterable upgrade over this chapter's `eprintln!` diagnostics, plus error-plus-backtrace via `eyre` |
| 10 | Structs, OOP and Traits | your own types — including the `PartialEq`/`Debug` derives this chapter's macros kept demanding |
| 11 | Errors in Depth | real error types for the panics you learned to catch here |

**Carry forward.** You should now be able to do these without looking anything up:

- choose between `assert!`, `assert_eq!`, `debug_assert!` and `matches!` and say why
- pin a layout or an arithmetic invariant with a const assert and read E0080 when it fails
- extract a panic payload with the String-then-&str downcasts
- recite the niche table and predict `Option`'s size cost before measuring
- name a type for humans and compare types for identity, with the right tool each time
